# Dados de Anomalias - Detecção de Anomalias

Este notebook demonstra a criação e análise de dados com anomalias pontuais.

## Objetivo

Aprender a criar dados simulados com anomalias e aplicar técnicas de detecção usando Isolation Forest.


## Instalação das Bibliotecas


In [ ]:
%pip install numpy matplotlib scikit-learn


## Criação dos Dados

Simula medições de pH normais e adiciona anomalias pontuais para análise.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

np.random.seed(42)

# Criar dados normais de pH (distribuição normal em torno de 7.0)
ph_normais = np.random.normal(loc=7.0, scale=0.3, size=100)

# Adicionar anomalias pontuais (valores extremos de pH)
anomalias_pontuais = np.array([2.0, 11.0])

# Combinar todos os dados
dados_ph = np.concatenate([ph_normais, anomalias_pontuais])

print(f"Total de amostras: {len(dados_ph)}")
print(f"Amostras normais: {len(ph_normais)}")
print(f"Anomalias: {len(anomalias_pontuais)}")
print(f"\nValores de pH:")
print(f"Mínimo: {dados_ph.min():.2f}")
print(f"Máximo: {dados_ph.max():.2f}")
print(f"Média: {dados_ph.mean():.2f}")
print(f"Desvio padrão: {dados_ph.std():.2f}")


## Visualização dos Dados


In [ ]:
plt.figure(figsize=(12, 6))

# Gráfico 1: Histograma
plt.subplot(1, 2, 1)
plt.hist(ph_normais, bins=20, alpha=0.7, label='pH Normais', color='blue')
plt.hist(anomalias_pontuais, bins=2, alpha=0.7, label='Anomalias', color='red')
plt.xlabel('pH')
plt.ylabel('Frequência')
plt.title('Distribuição de Valores de pH')
plt.legend()
plt.grid(True, alpha=0.3)

# Gráfico 2: Box Plot
plt.subplot(1, 2, 2)
plt.boxplot(dados_ph, vert=True)
plt.ylabel('pH')
plt.title('Box Plot - Valores de pH')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAs anomalias (pH 2.0 e 11.0) são claramente visíveis como valores extremos!")


## Detecção de Anomalias usando Isolation Forest

O Isolation Forest é um algoritmo eficiente para detectar anomalias baseado no princípio de que anomalias são poucas e diferentes.


In [ ]:
# Preparar dados para o modelo (precisa ser 2D)
X = dados_ph.reshape(-1, 1)

# Criar e treinar o modelo Isolation Forest
# contamination: proporção esperada de anomalias (2/102 ≈ 0.02)
modelo = IsolationForest(contamination=0.02, random_state=42)
predicoes = modelo.fit_predict(X)

# Separar normais (-1) e anomalias (1)
# Isolation Forest retorna -1 para anomalias e 1 para normais
anomalias_detectadas = dados_ph[predicoes == -1]
normais_detectados = dados_ph[predicoes == 1]

print(f"Anomalias detectadas: {len(anomalias_detectadas)}")
print(f"Valores das anomalias: {anomalias_detectadas}")
print(f"\nNormais detectados: {len(normais_detectados)}")
print(f"\nAnomalias reais: {anomalias_pontuais}")
print(f"Anomalias detectadas pelo modelo: {anomalias_detectadas}")


## Visualização dos Resultados


In [ ]:
plt.figure(figsize=(14, 6))

# Gráfico 1: Dados originais
plt.subplot(1, 2, 1)
plt.scatter(range(len(ph_normais)), ph_normais, alpha=0.6, label='pH Normais', color='blue', s=50)
plt.scatter(range(len(ph_normais), len(ph_normais) + len(anomalias_pontuais)), 
           anomalias_pontuais, alpha=0.9, label='Anomalias Reais', color='red', s=150, marker='X')
plt.xlabel('Índice da Amostra')
plt.ylabel('pH')
plt.title('Dados Originais')
plt.legend()
plt.grid(True, alpha=0.3)

# Gráfico 2: Resultados da detecção
plt.subplot(1, 2, 2)
indices_normais = np.where(predicoes == 1)[0]
indices_anomalias = np.where(predicoes == -1)[0]

plt.scatter(indices_normais, dados_ph[indices_normais], 
           alpha=0.6, label='Normal', color='blue', s=50)
plt.scatter(indices_anomalias, dados_ph[indices_anomalias], 
           alpha=0.9, label='Anomalia Detectada', color='red', s=150, marker='X')
plt.xlabel('Índice da Amostra')
plt.ylabel('pH')
plt.title('Resultados da Detecção (Isolation Forest)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calcular acurácia
anomalias_reais_indices = list(range(100, 102))
anomalias_detectadas_indices = list(indices_anomalias)

print("\n" + "="*60)
print("AVALIAÇÃO DA DETECÇÃO")
print("="*60)
print(f"Anomalias reais (índices): {anomalias_reais_indices}")
print(f"Anomalias detectadas (índices): {anomalias_detectadas_indices}")
print(f"\nAcurácia: {len(set(anomalias_reais_indices) & set(anomalias_detectadas_indices)) / len(anomalias_reais_indices) * 100:.1f}%")


## Interpretação dos Resultados

**Isolation Forest** funciona bem para detectar anomalias de ponto porque:

1. **Isola anomalias rapidamente**: Anomalias são diferentes e poucas, então são fáceis de isolar
2. **Não precisa de dados rotulados**: É um método não-supervisionado
3. **Eficiente**: Funciona bem mesmo com muitos dados

**Neste exemplo:**
- Criamos 100 amostras normais de pH (em torno de 7.0)
- Adicionamos 2 anomalias (pH 2.0 e 11.0)
- O modelo conseguiu detectar ambas as anomalias corretamente

**Aplicações práticas:**
- Detecção de fraudes
- Monitoramento de qualidade
- Detecção de intrusões em sistemas
- Análise de dados médicos
